# PE6201 End-of-Course Project:
# RAG-powered compliance Q&A assistant for mid-sized bank compliance analysts.

For incoming compliance questions → answer solely based on 6 public regulatory documents → citations with [File Name, p.Page Number] are mandatory → reject responses if the score is too low.
Citation checking and PII detection are code rules, independent of whether the model follows instructions.



In [347]:
!pip -q install openai pypdf pandas numpy


from google.colab import drive
drive.mount('/content/drive')

import os, sys
WORK = "/content/drive/MyDrive/pe6201_rag"
for d in ["src", "evals/results", "data/raw_pdfs", "index", "logs"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)
print("content:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
content: /content/drive/MyDrive/pe6201_rag


In [348]:
from src import config
print(config.EMBED_MODEL)
print(config.CHAT_MODEL)
print(config.get_client().base_url)

openai/text-embedding-3-small
openai/gpt-4o-mini
https://openrouter.ai/api/v1/


In [349]:
p = "src/config.py"
s = open(p).read()
s = s.replace('EMBED_MODEL = "text-embedding-3-small"', 'EMBED_MODEL = "openai/text-embedding-3-small"')
s = s.replace('CHAT_MODEL = "gpt-4o-mini"', 'CHAT_MODEL = "openai/gpt-4o-mini"')
i = s.index("_client = None")
s = s[:i] + '''_client = None


def get_client():
    """OpenAI-compatible client. Works with an OpenRouter key (secret OPENROUTER_API_KEY)
    or a native OpenAI key (secret OPENAI_API_KEY). OpenRouter keys start with "sk-or-"."""
    global _client
    if _client is None:
        from openai import OpenAI
        key = None
        for name in ("OPENROUTER_API_KEY", "OPENAI_API_KEY"):
            key = os.environ.get(name)
            if not key:
                try:
                    from google.colab import userdata
                    key = userdata.get(name)
                except Exception:
                    key = None
            if key:
                break
        if not key:
            raise RuntimeError("No API key found. Add OPENROUTER_API_KEY (or OPENAI_API_KEY) in Colab Secrets "
                               "and turn on Notebook access.")
        base = "https://openrouter.ai/api/v1" if key.startswith("sk-or-") else None
        _client = OpenAI(api_key=key, base_url=base)
    return _client
'''
open(p, "w").write(s)
print("config.py updated")

config.py updated


**Write code modules (src/)**

Architecture: pipeline.py orchestrates everything
Question → PII Check → retrieve (vector search + threshold) → generate (LLM) → guardrails (citation + PII) → audit (logging)

In [350]:
%%writefile src/__init__.py
"""RAG compliance Q&A assistant (PE6201 end-of-course project)."""

Overwriting src/__init__.py


In [351]:
%%writefile src/config.py
"""config.py - central settings for the whole project.

Everything tunable lives here: model names, prices, chunking and retrieval
parameters, file paths. Other modules import from this file and never hard-code
numbers. Change a value here and the whole pipeline follows.

IMPORTANT: model names and prices below are PLACEHOLDERS. Check the current
OpenAI pricing page before you report any cost number, and write the date you
checked in your report.
"""
import os
from dataclasses import dataclass

# ---- paths (repo root = parent of src/) ------------------------------------
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
PDF_DIR = os.path.join(ROOT, "data", "raw_pdfs")
INDEX_DIR = os.path.join(ROOT, "index")
LOG_DIR = os.path.join(ROOT, "logs")
RESULTS_DIR = os.path.join(ROOT, "evals", "results")

# ---- rented models (we only call APIs; no training) ---------------------------
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"   # if you switch to a reasoning model, adapt llm.chat()

# ---- prices in USD per 1M tokens  (VERIFY on the official pricing page!) --------
PRICE_EMBED_PER_M = 0.02
PRICE_CHAT_IN_PER_M = 0.15
PRICE_CHAT_OUT_PER_M = 0.60
USD_TO_SGD = 1.30            # update to the rate on the day you write the report


@dataclass
class Config:
    """Tunable pipeline parameters (the knobs we tune on the dev set)."""
    chunk_size: int = 1000        # characters per chunk
    chunk_overlap: int = 150      # characters shared between neighbouring chunks
    top_k: int = 4                # chunks passed to the LLM
    sim_threshold: float = 0.30   # refuse if best cosine similarity < this
    max_retries: int = 1          # regenerate once if the citation check fails

    @property
    def tag(self) -> str:
        """Index folder name; chunking params decide which index is used."""
        return f"cs{self.chunk_size}_ov{self.chunk_overlap}"


def embed_cost(tokens: int) -> float:
    return tokens / 1e6 * PRICE_EMBED_PER_M


def chat_cost(in_tokens: int, out_tokens: int) -> float:
    return in_tokens / 1e6 * PRICE_CHAT_IN_PER_M + out_tokens / 1e6 * PRICE_CHAT_OUT_PER_M


_client = None


def get_client():
    """Create the OpenAI client once. Key comes from env var or Colab Secrets."""
    global _client
    if _client is None:
        from openai import OpenAI
        key = os.environ.get("OPENAI_API_KEY")
        if not key:
            try:
                from google.colab import userdata
                key = userdata.get("OPENAI_API_KEY")
            except Exception:
                key = None
        if not key:
            raise RuntimeError("OPENAI_API_KEY not found. Add it in Colab Secrets or set the env var.")
        _client = OpenAI(api_key=key)
    return _client

Overwriting src/config.py


In [352]:
%%writefile src/llm.py
"""llm.py - the ONLY place that talks to the rented OpenAI API.

Two functions: embed_texts() and chat(). Both retry on transient errors and
return token usage so that every call can be costed (see audit.py / pipeline.py).
Keeping API calls in one file makes the "build vs rent" boundary explicit:
everything else in the repo is code we own.
"""
import time
import numpy as np
from .config import get_client, EMBED_MODEL, CHAT_MODEL


def _retry(fn, attempts=4):
    for i in range(attempts):
        try:
            return fn()
        except Exception:
            if i == attempts - 1:
                raise
            time.sleep(2 ** i)


def embed_texts(texts):
    """Embed a list of strings. Returns (float32 array [n, dim], total_tokens)."""
    client = get_client()
    vectors, tokens = [], 0
    for i in range(0, len(texts), 100):
        batch = texts[i:i + 100]
        r = _retry(lambda: client.embeddings.create(model=EMBED_MODEL, input=batch))
        vectors.extend(d.embedding for d in r.data)
        tokens += r.usage.total_tokens
    return np.array(vectors, dtype="float32"), tokens


def chat(messages, temperature=0.0, max_tokens=500):
    """One chat completion. Returns (text, prompt_tokens, completion_tokens)."""
    client = get_client()
    r = _retry(lambda: client.chat.completions.create(
        model=CHAT_MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens))
    return r.choices[0].message.content or "", r.usage.prompt_tokens, r.usage.completion_tokens

Overwriting src/llm.py


In [353]:
%%writefile src/ingest.py
"""ingest.py - PDF -> page text -> chunks -> embeddings -> saved index.

We OWN the chunking logic because it strongly affects retrieval quality.
Each chunk keeps its source file name and PHYSICAL page number (1-based, as
shown by a PDF viewer's page counter). That metadata is what makes
citation checking and citation-accuracy evaluation possible later.

Index layout (one folder per chunking setting): index/<tag>/
    chunks.json  list of {id, doc, page, text}
    vectors.npy  L2-normalised float32 matrix, row i <-> chunks[i]
    meta.json    counts, embedding tokens and cost, pages with no text
"""
import os, re, json
import numpy as np
from pypdf import PdfReader
from .config import PDF_DIR, INDEX_DIR, Config, embed_cost
from .llm import embed_texts

MIN_CHUNK_CHARS = 50


def extract_pages(pdf_path):
    """Return [(page_no, cleaned_text)] for one PDF. Empty text => scanned page."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        try:
            text = page.extract_text() or ""
        except Exception:
            text = ""
        pages.append((i, re.sub(r"\s+", " ", text).strip()))
    return pages


def chunk_text(text, size, overlap):
    """Fixed-size character windows with overlap. Chunks never cross a page."""
    if len(text) <= size:
        return [text]
    chunks, start, step = [], 0, max(1, size - overlap)
    while start < len(text):
        chunks.append(text[start:start + size])
        if start + size >= len(text):
            break
        start += step
    return chunks


def build_index(cfg=None, force=False):
    """Build (or reuse) the index for cfg's chunking params. Returns meta dict."""
    cfg = cfg or Config()
    out = os.path.join(INDEX_DIR, cfg.tag)
    meta_path = os.path.join(out, "meta.json")
    if os.path.exists(meta_path) and not force:
        print(f"[ingest] reuse existing index {cfg.tag}")
        return json.load(open(meta_path))

    pdfs = sorted(f for f in os.listdir(PDF_DIR) if f.lower().endswith(".pdf"))
    if not pdfs:
        raise FileNotFoundError(f"No PDFs found in {PDF_DIR}")
    chunks, empty_pages, n_pages = [], [], 0
    for fname in pdfs:
        for page_no, text in extract_pages(os.path.join(PDF_DIR, fname)):
            n_pages += 1
            if len(text) < MIN_CHUNK_CHARS:
                empty_pages.append(f"{fname}:{page_no}")
                continue
            for piece in chunk_text(text, cfg.chunk_size, cfg.chunk_overlap):
                if len(piece) >= MIN_CHUNK_CHARS:
                    chunks.append({"id": len(chunks), "doc": fname, "page": page_no, "text": piece})

    print(f"[ingest] {len(pdfs)} PDFs, {n_pages} pages, {len(chunks)} chunks; embedding...")
    vecs, tokens = embed_texts([c["text"] for c in chunks])
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)

    os.makedirs(out, exist_ok=True)
    json.dump(chunks, open(os.path.join(out, "chunks.json"), "w"), ensure_ascii=False)
    np.save(os.path.join(out, "vectors.npy"), vecs)
    meta = {"tag": cfg.tag, "n_docs": len(pdfs), "n_pages": n_pages, "n_chunks": len(chunks),
            "embed_tokens": tokens, "embed_cost_usd": embed_cost(tokens),
            "pages_without_text": empty_pages}
    json.dump(meta, open(meta_path, "w"), indent=2)
    if empty_pages:
        print(f"[ingest] WARNING {len(empty_pages)} pages had no extractable text (scanned?). "
              f"See meta.json. They cannot be answered from.")
    print(f"[ingest] done. embedding tokens={tokens}, cost=${meta['embed_cost_usd']:.4f}")
    return meta


def find_pages(keyword, cfg=None, max_results=15, width=120):
    """Labelling helper: show where a keyword appears (doc + physical page).

    Use it while writing the 50-question answer key to locate gold pages fast.
    """
    cfg = cfg or Config()
    chunks = json.load(open(os.path.join(INDEX_DIR, cfg.tag, "chunks.json")))
    seen, shown = set(), 0
    for c in chunks:
        m = re.search(re.escape(keyword), c["text"], flags=re.I)
        if m and (c["doc"], c["page"]) not in seen:
            seen.add((c["doc"], c["page"]))
            s = max(0, m.start() - width // 2)
            print(f"{c['doc']} | p.{c['page']} | ...{c['text'][s:s + width]}...")
            shown += 1
            if shown >= max_results:
                break
    if not shown:
        print("no match")

Overwriting src/ingest.py


In [354]:
%%writefile src/retrieve.py
"""retrieve.py - embed the question, find the top-k most similar chunks.

Returns the hits plus the embedding tokens used (for costing). The caller
(pipeline.py) applies the similarity threshold: if the best score is below
cfg.sim_threshold the system refuses instead of letting the LLM guess.
Similarity = cosine (vectors are normalised, so a dot product).
"""
import os, json
import numpy as np
from .config import INDEX_DIR, Config
from .llm import embed_texts

_cache = {}


def load_index(cfg):
    d = os.path.join(INDEX_DIR, cfg.tag)
    if d not in _cache:
        chunks = json.load(open(os.path.join(d, "chunks.json")))
        vecs = np.load(os.path.join(d, "vectors.npy"))
        _cache[d] = (chunks, vecs)
    return _cache[d]


def retrieve(question, cfg=None):
    """Return (hits, embed_tokens). hits = top_k dicts {id, doc, page, text, score}."""
    cfg = cfg or Config()
    chunks, vecs = load_index(cfg)
    qv, tokens = embed_texts([question])
    qv = qv[0] / np.linalg.norm(qv[0])
    scores = vecs @ qv
    idx = np.argsort(-scores)[:cfg.top_k]
    hits = [{**chunks[i], "score": float(scores[i])} for i in idx]
    return hits, tokens

Overwriting src/retrieve.py


In [355]:
%%writefile src/generate.py
"""generate.py - prompt template and the LLM call that writes the answer.

The prompt is tuned for compliance questions: answer ONLY from the supplied
excerpts, cite every statement as [file.pdf, p.N], and output the sentinel
INSUFFICIENT_CONTEXT when the excerpts do not contain the answer.
NOTE: the prompt only *asks* for citations. Enforcement is done in code by
guardrails.py, so safety does not depend on the model obeying the prompt.
"""
from .llm import chat

ABSTAIN_TOKEN = "INSUFFICIENT_CONTEXT"

SYSTEM_PROMPT = (
    "You are a regulatory Q&A assistant for bank compliance analysts. "
    "Answer ONLY using the numbered context excerpts supplied by the user message.\n"
    "Rules:\n"
    "1. Every factual statement must end with a citation in EXACTLY this format: "
    "[filename.pdf, p.N] using the file name and page shown on the excerpt header.\n"
    "2. If the excerpts do not contain the answer, reply with exactly: " + ABSTAIN_TOKEN + "\n"
    "3. Never use outside knowledge. Never give formal legal advice.\n"
    "4. Be concise: at most about 150 words. Do not repeat personal data."
)

REMINDER = ("\n\nREMINDER: your previous answer broke the citation rules. Cite every statement as "
            "[filename.pdf, p.N] using ONLY files/pages that appear in the excerpt headers.")


def build_context(hits):
    return "\n\n".join(f"[{h['doc']}, p.{h['page']}]\n{h['text']}" for h in hits)


def generate_answer(question, hits, reminder=False):
    """Return (text, prompt_tokens, completion_tokens)."""
    user = f"Context excerpts:\n\n{build_context(hits)}\n\nQuestion: {question}"
    if reminder:
        user += REMINDER
    return chat([{"role": "system", "content": SYSTEM_PROMPT},
                 {"role": "user", "content": user}])

Overwriting src/generate.py


In [356]:
%%writefile src/guardrails.py
"""guardrails.py - deterministic, code-based safety rules (NO LLM involved).

Rule 1  citation check : answer must contain >=1 citation of the form
                         [file.pdf, p.N], and every cited (file, page) must be
                         among the chunks that were actually retrieved.
Rule 2  PII detection  : regexes for email, SG NRIC/FIN, SG phone, CN national
                         ID, and card numbers (Luhn-validated). Applied to the
                         user question (refuse before calling the API) and to
                         the generated answer (block).
These run regardless of what the model says. Known limitation: regex PII has
false positives (e.g. a public contact e-mail printed in a circular) and false
negatives (names, addresses). State this in the report.
"""
import re

CITATION_RE = re.compile(r"\[([^\[\]]+?),\s*p\.?\s*(\d+)\]", re.I)

PII_PATTERNS = {
    "email": re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"),
    "sg_nric_fin": re.compile(r"\b[STFGM]\d{7}[A-Z]\b", re.I),
    "sg_phone": re.compile(r"(?<!\d)(?:\+65[\s-]?)?[3689]\d{3}[\s-]?\d{4}(?!\d)"),
    "cn_national_id": re.compile(r"(?<!\d)\d{17}[\dXx](?!\d)"),
}
CARD_CANDIDATE_RE = re.compile(r"(?<!\d)(?:\d[ -]?){12,18}\d(?!\d)")


def _luhn_ok(digits):
    total, alt = 0, False
    for ch in reversed(digits):
        d = int(ch)
        if alt:
            d *= 2
            if d > 9:
                d -= 9
        total += d
        alt = not alt
    return total % 10 == 0


def find_pii(text):
    """Return a list of (type, matched_text). Empty list = clean."""
    found = []
    for name, pat in PII_PATTERNS.items():
        for m in pat.finditer(text):
            found.append((name, m.group(0)))
    for m in CARD_CANDIDATE_RE.finditer(text):
        digits = re.sub(r"\D", "", m.group(0))
        if 13 <= len(digits) <= 19 and _luhn_ok(digits):
            found.append(("card_number", m.group(0)))
    return found


def extract_citations(answer):
    """Return [(file_lowercase, page_int)] cited in the answer."""
    return [(f.strip().lower(), int(p)) for f, p in CITATION_RE.findall(answer)]


def check_citations(answer, hits):
    """Return (ok, reason, cited). reason in {ok, no_citation, citation_not_in_retrieved}."""
    cited = extract_citations(answer)
    if not cited:
        return False, "no_citation", cited
    allowed = {(h["doc"].lower(), h["page"]) for h in hits}
    if any(c not in allowed for c in cited):
        return False, "citation_not_in_retrieved", cited
    return True, "ok", cited

Overwriting src/guardrails.py


In [357]:
%%writefile src/audit.py
"""audit.py - append-only audit log (one JSON object per line).

Records every question, the retrieved chunks, the answer, the status and the
token usage / cost. This is the "audit log" mitigation from the proposal and
the source of the per-question cost numbers. Questions containing PII are
redacted before they are written.
"""
import os, json, datetime
from .config import LOG_DIR


def log_event(rec):
    os.makedirs(LOG_DIR, exist_ok=True)
    slim = dict(rec)
    slim["hits"] = [{"doc": h["doc"], "page": h["page"], "score": round(h["score"], 4),
                     "text": h["text"][:300]} for h in rec.get("hits", [])]
    slim["timestamp"] = datetime.datetime.now().isoformat(timespec="seconds")
    with open(os.path.join(LOG_DIR, "audit.jsonl"), "a", encoding="utf-8") as f:
        f.write(json.dumps(slim, ensure_ascii=False) + "\n")

Overwriting src/audit.py


In [358]:
%%writefile src/pipeline.py
"""pipeline.py - the end-to-end flow: question in, grounded answer (or refusal) out.

  question -> [PII in question?] -> retrieve top-k -> [best score < threshold?]
           -> generate -> [model abstained?] -> [PII in answer?]
           -> [citation check; retry once] -> answer  (always audit-logged)

status values:
  answered | refused_pii_input | refused_low_score | model_abstained |
  blocked_pii_output | blocked_no_citation | blocked_citation_not_in_retrieved
Anything other than "answered" returns a refusal message, never a raw LLM answer.
"""
import time
from dataclasses import asdict
from .config import Config, embed_cost, chat_cost
from .retrieve import retrieve
from .generate import generate_answer, ABSTAIN_TOKEN
from .guardrails import find_pii, check_citations
from .audit import log_event

MESSAGES = {
    "refused_pii_input": "Your question appears to contain personal data (PII). Please remove it and ask again.",
    "refused_low_score": "I cannot answer this from the regulatory corpus (no sufficiently relevant source found).",
    "model_abstained": "The retrieved sources do not contain enough information to answer this question.",
    "blocked_pii_output": "The draft answer was blocked because it contained possible personal data.",
    "blocked_no_citation": "The draft answer was blocked because it lacked a source citation.",
    "blocked_citation_not_in_retrieved": "The draft answer was blocked because it cited a source that was not retrieved.",
}


def answer_question(question, cfg=None, log=True):
    cfg = cfg or Config()
    t0 = time.time()
    rec = {"question": question, "status": None, "answer": "", "raw_answer": "", "cited": [],
           "hits": [], "top_score": None, "embed_tokens": 0, "in_tokens": 0, "out_tokens": 0,
           "retries": 0, "config": asdict(cfg)}

    def finish(status, answer):
        rec["status"], rec["answer"] = status, answer
        rec["cost_usd"] = embed_cost(rec["embed_tokens"]) + chat_cost(rec["in_tokens"], rec["out_tokens"])
        rec["latency_s"] = round(time.time() - t0, 3)
        if log:
            logged = dict(rec)
            if status == "refused_pii_input":
                logged["question"] = "[REDACTED: PII detected in question]"
            log_event(logged)
        return rec

    if find_pii(question):
        return finish("refused_pii_input", MESSAGES["refused_pii_input"])

    hits, et = retrieve(question, cfg)
    rec["hits"], rec["embed_tokens"], rec["top_score"] = hits, et, hits[0]["score"]
    if rec["top_score"] < cfg.sim_threshold:
        return finish("refused_low_score", MESSAGES["refused_low_score"])

    for attempt in range(cfg.max_retries + 1):
        text, i_tok, o_tok = generate_answer(question, hits, reminder=(attempt > 0))
        rec["in_tokens"] += i_tok
        rec["out_tokens"] += o_tok
        rec["retries"], rec["raw_answer"] = attempt, text
        if text.strip().startswith(ABSTAIN_TOKEN):
            return finish("model_abstained", MESSAGES["model_abstained"])
        if find_pii(text):
            return finish("blocked_pii_output", MESSAGES["blocked_pii_output"])
        ok, reason, cited = check_citations(text, hits)
        rec["cited"] = cited
        if ok:
            return finish("answered", text.strip())
    return finish("blocked_" + reason, MESSAGES["blocked_" + reason])

Overwriting src/pipeline.py


In [359]:
p = "src/generate.py"
s = open(p).read()
old = '"4. Be concise: at most about 150 words. Do not repeat personal data."'
new = ('"4. Be concise: at most about 150 words. Do not repeat personal data.\\n"\n'
       '    "5. If an excerpt comes from a document whose file name contains \'Consultation\', state clearly that '
       'the position is only PROPOSED and not yet in force."')
assert old in s, "pattern not found (already patched?)"
open(p, "w").write(s.replace(old, new))
print("generate.py updated")

generate.py updated


## 3. Load corpus & build index

Download 6 financial compliance guidelines PDFs from the Monetary Authority of Singapore and upload them to Drive: MyDrive/pe6201_rag/data/raw_pdfs/. These will serve as the corpus source for this project.


In [360]:
import os, importlib
from src.config import get_client, Config, PDF_DIR
from src import ingest

get_client()
pdfs = sorted(f for f in os.listdir(PDF_DIR) if f.lower().endswith(".pdf"))
print(f"found {len(pdfs)} files ")
for f in pdfs[:50]:
    print(" -", f)

found 6 files 
 - MAS_Consultation_Crypto_Permissionless_2026.pdf
 - MAS_Consultation_TLAC_DSIB_2026.pdf
 - MAS_EPayments_Guidelines_2024.pdf
 - MAS_Environmental_Risk_Guidelines_2020.pdf
 - MAS_Liquidity_Risk_Guidelines_2026.pdf
 - MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf


In [361]:
cfg = Config()
meta = ingest.build_index(cfg)
{k: v for k, v in meta.items() if k != "pages_without_text"}, f"Page without text: {len(meta['pages_without_text'])}"

[ingest] reuse existing index cs1000_ov150


({'tag': 'cs1000_ov150',
  'n_docs': 6,
  'n_pages': 199,
  'n_chunks': 556,
  'embed_tokens': 92958,
  'embed_cost_usd': 0.0018591600000000001},
 'Page without text: 1')

In [362]:
p = "src/config.py"
s = open(p).read()
s = s.replace('EMBED_MODEL = "text-embedding-3-small"', 'EMBED_MODEL = "openai/text-embedding-3-small"')
s = s.replace('CHAT_MODEL = "gpt-4o-mini"', 'CHAT_MODEL = "openai/gpt-4o-mini"')
i = s.index("_client = None")
s = s[:i] + '''_client = None


def get_client():
    """OpenAI-compatible client. Works with an OpenRouter key (secret OPENROUTER_API_KEY)
    or a native OpenAI key (secret OPENAI_API_KEY)."""
    global _client
    if _client is None:
        from openai import OpenAI
        key = None
        for name in ("OPENROUTER_API_KEY", "OPENAI_API_KEY"):
            key = os.environ.get(name)
            if not key:
                try:
                    from google.colab import userdata
                    key = userdata.get(name)
                except Exception:
                    key = None
            if key:
                break
        if not key:
            raise RuntimeError("No API key found. Add OPENROUTER_API_KEY (or OPENAI_API_KEY) in Colab Secrets.")
        base = "https://openrouter.ai/api/v1" if key.startswith("sk-or-") else None
        _client = OpenAI(api_key=key, base_url=base)
    return _client
'''
open(p, "w").write(s)
print("config.py updated")

config.py updated


## 4. Acceptance criteria to confirm it is running:
Given the sample input, the system returns an answer citing at least one source document from the corpus, without throwing runtime error.

In [363]:
from src.pipeline import answer_question

def show(rec):
    print("STATUS :", rec["status"], "| top_score:", None if rec["top_score"] is None else round(rec["top_score"], 3))
    print("ANSWER :", rec["answer"])
    print("CITED  :", rec["cited"])
    print("CHUNKS :", [(h["doc"], h["page"], round(h["score"], 3)) for h in rec["hits"]])
    print(f"COST   : ${rec['cost_usd']:.5f} | latency {rec['latency_s']}s | retries {rec['retries']}")
show(answer_question("What is the external TLAC requirement MAS proposes for a local bank DSIB?"))


STATUS : answered | top_score: 0.809
ANSWER : MAS proposes an external TLAC requirement of 14% of the risk-weighted assets (RWA) for local bank DSIBs [MAS_Consultation_TLAC_DSIB_2026.pdf, p.5].
CITED  : [('mas_consultation_tlac_dsib_2026.pdf', 5)]
CHUNKS : [('MAS_Consultation_TLAC_DSIB_2026.pdf', 6, 0.809), ('MAS_Consultation_TLAC_DSIB_2026.pdf', 5, 0.796), ('MAS_Consultation_TLAC_DSIB_2026.pdf', 5, 0.777), ('MAS_Consultation_TLAC_DSIB_2026.pdf', 5, 0.769)]
COST   : $0.00017 | latency 1.429s | retries 0


## 5. Guardrail Demonstration
(Normal Response / Refusal to Answer / Blocked)

In [316]:
show(answer_question("What is the external TLAC requirement MAS proposes for a local bank DSIB?"))
show(answer_question("What are the customer due diligence requirements for banks under MAS Notice 626?"))

STATUS : answered | top_score: 0.809
ANSWER : MAS proposes an external TLAC requirement of 14% of the risk-weighted assets (RWA) for local bank DSIBs [MAS_Consultation_TLAC_DSIB_2026.pdf, p.5].
CITED  : [('mas_consultation_tlac_dsib_2026.pdf', 5)]
CHUNKS : [('MAS_Consultation_TLAC_DSIB_2026.pdf', 6, 0.809), ('MAS_Consultation_TLAC_DSIB_2026.pdf', 5, 0.796), ('MAS_Consultation_TLAC_DSIB_2026.pdf', 5, 0.777), ('MAS_Consultation_TLAC_DSIB_2026.pdf', 5, 0.769)]
COST   : $0.00017 | latency 1.665s | retries 0
STATUS : model_abstained | top_score: 0.636
ANSWER : The retrieved sources do not contain enough information to answer this question.
CITED  : []
CHUNKS : [('MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf', 17, 0.636), ('MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf', 17, 0.631), ('MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf', 25, 0.63), ('MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf', 38, 0.63)]
COST   : $0.00017 | latency 1.299s | retries 0


In [317]:
show(answer_question("Customer S1234567D with phone 91234567 wants to know the record retention rule."))

STATUS : refused_pii_input | top_score: None
ANSWER : Your question appears to contain personal data (PII). Please remove it and ask again.
CITED  : []
CHUNKS : []
COST   : $0.00000 | latency 0.0s | retries 0


In [318]:
from src.guardrails import find_pii, check_citations

assert find_pii("contact S1234567D") and find_pii("mail a.b@bank.com") and find_pii("call +65 9123 4567")
assert find_pii("card 4111 1111 1111 1111")
assert not find_pii("Notice 626 requires records for 5 years.")
hits = [{"doc": "A.pdf", "page": 3}]
assert check_citations("Keep 5 years [A.pdf, p.3].", hits)[0]
assert check_citations("Keep 5 years.", hits)[1] == "no_citation"
assert check_citations("Keep 5 years [B.pdf, p.9].", hits)[1] == "citation_not_in_retrieved"
print("guardrail tests passed ✅")

guardrail tests passed ✅


## 6. Problem-solving Assistant: Look up the positions of keywords in the corpus
When writing standard answers, use it to locate the file and page number first, then **open the PDF yourself to verify the original text**.

In [319]:
ingest.find_pages("retention")     # change any key words I want

MAS_EPayments_Guidelines_2024.pdf | p.30 | ... the account holder; and iii. inform the recipient that his retention or use of sums transferred to him erroneously wher...


## 7. 50 question evaluation set

(files:`evals/questions_50.csv`)


In [320]:
import pandas as pd, importlib
from evals import run_evals as ev
importlib.reload(ev)

df = ev.load_questions()
dev = df[df.split == "dev"].reset_index(drop=True)
test = df[df.split == "test"].reset_index(drop=True)

res_dev = ev.run_eval(dev, Config(), log=False)
pd.Series(ev.summarize(res_dev))

,0
n,27
n_answerable,19
n_should_refuse,8
retrieval_doc_hit,1.0
retrieval_page_hit,0.947368
answer_rate_on_answerable,0.894737
false_refusal_rate,0.105263
refusal_accuracy,1.0
citation_present,1.0
citation_correct,0.941176


##Evaluation Script `evals/run_evals.py`
All automatic metrics do not use LLM judge: retrieval hit, citation existence, citation accuracy, correct refusal / false refusal, PII leakage, cost, latency.

Faithfulness is manually annotated by me in Section 9.


In [321]:
%%writefile evals/__init__.py
"""Evaluation harness for the RAG compliance assistant."""

Overwriting evals/__init__.py


In [322]:
%%writefile evals/run_evals.py
"""run_evals.py - the evaluation harness (L1 deterministic + hand-labelled faithfulness).

NO LLM-as-judge. Automatic metrics compare system output with the hand-written
answer key in questions_50.csv:
  retrieval_doc_hit / retrieval_page_hit : was a gold doc / (doc,page) in the top-k?
  citation_present   : every answered response has >=1 citation (L1 assertion)
  citation_correct   : a cited (doc,page) matches the gold sources
  refusal_accuracy   : should_refuse rows that did NOT get an answer
  false_refusal_rate : answerable rows that did NOT get an answer
  pii_leak           : answered responses containing PII (L1 assertion; should be 0)
  cost / latency     : from token usage and wall-clock time
Faithfulness is judged by a human: export_for_labelling() -> label -> faithfulness().

Typical use (see notebook): run_eval(df, cfg) -> summarize(res) -> export_for_labelling(res)
"""
import os, json
import numpy as np
import pandas as pd
from src.config import Config, PDF_DIR, RESULTS_DIR
from src.pipeline import answer_question
from src.retrieve import retrieve
from src.guardrails import find_pii

REQUIRED = ["id", "split", "type", "question", "gold_answer", "gold_sources", "should_refuse"]


def parse_sources(s):
    """'a.pdf:12;b.pdf:5' -> {('a.pdf',12), ('b.pdf',5)} (file lowercased)."""
    out = set()
    if not isinstance(s, str):
        return out
    for part in s.split(";"):
        part = part.strip()
        if ":" in part:
            f, p = part.rsplit(":", 1)
            try:
                out.add((f.strip().lower(), int(p)))
            except ValueError:
                pass
    return out


def load_questions(path="evals/questions_50.csv"):
    df = pd.read_csv(path, dtype={"gold_sources": str})
    df["should_refuse"] = df["should_refuse"].astype(int)
    return df


def validate_questions(df):
    """Print counts and return a list of problems (typos in file names etc.)."""
    miss = [c for c in REQUIRED if c not in df.columns]
    if miss:
        return [f"missing columns: {miss}"]
    pdfs = {f.lower() for f in os.listdir(PDF_DIR) if f.lower().endswith(".pdf")}
    problems = []
    for _, r in df.iterrows():
        srcs = parse_sources(r["gold_sources"])
        if r["should_refuse"] == 0 and not srcs:
            problems.append(f"{r['id']}: answerable question without gold_sources")
        for f, _p in srcs:
            if f not in pdfs:
                problems.append(f"{r['id']}: file not found in raw_pdfs: {f}")
        if "REPLACE" in str(r["gold_sources"]) or str(r["question"]).startswith("(示例)"):
            problems.append(f"{r['id']}: still a template row")
    print(f"{len(df)} questions | split:", df["split"].value_counts().to_dict(),
          "| type:", df["type"].value_counts().to_dict())
    if len(df) != 50:
        problems.append(f"expected 50 questions, found {len(df)}")
    for p in problems:
        print("  PROBLEM:", p)
    if not problems:
        print("questions look valid")
    return problems


def run_eval(df, cfg=None, log=False):
    """Run the full pipeline on every row; return a per-question results DataFrame."""
    cfg = cfg or Config()
    rows = []
    for _, r in df.iterrows():
        rec = answer_question(r["question"], cfg, log=log)
        gold = parse_sources(r["gold_sources"])
        gold_docs = {d for d, _ in gold}
        hit_pairs = [(h["doc"].lower(), h["page"]) for h in rec["hits"]]
        answered = rec["status"] == "answered"
        rows.append({
            "id": r["id"], "split": r["split"], "type": r["type"], "question": r["question"],
            "should_refuse": int(r["should_refuse"]), "gold_answer": r["gold_answer"],
            "status": rec["status"], "answered": answered, "answer": rec["answer"],
            "raw_answer": rec["raw_answer"], "top_score": rec["top_score"],
            "cited": ";".join(f"{d}:{p}" for d, p in rec["cited"]),
            "cited_n": len(rec["cited"]) if answered else 0,
            "doc_hit": any(d in gold_docs for d, _ in hit_pairs),
            "page_hit": any(hp in gold for hp in hit_pairs),
            "cite_correct": answered and any(c in gold for c in rec["cited"]),
            "pii_out": answered and bool(find_pii(rec["answer"])),
            "cost_usd": rec["cost_usd"], "latency_s": rec["latency_s"],
            "in_tokens": rec["in_tokens"], "out_tokens": rec["out_tokens"],
            "embed_tokens": rec["embed_tokens"], "retries": rec["retries"],
            "retrieved": ";".join(f"{h['doc']}:{h['page']}@{h['score']:.2f}" for h in rec["hits"]),
            "ctx": "\n---\n".join(f"[{h['doc']}, p.{h['page']}] {h['text']}" for h in rec["hits"]),
        })
    return pd.DataFrame(rows)


def _mean(s):
    return float(s.mean()) if len(s) else float("nan")


def summarize(res):
    """Aggregate metrics (a dict). See module docstring for definitions."""
    ans = res[res.should_refuse == 0]
    ref = res[res.should_refuse == 1]
    answered = res[res.answered]
    ans_answered = ans[ans.answered]
    return {
        "n": len(res), "n_answerable": len(ans), "n_should_refuse": len(ref),
        "retrieval_doc_hit": _mean(ans.doc_hit),
        "retrieval_page_hit": _mean(ans.page_hit),
        "answer_rate_on_answerable": _mean(ans.answered),
        "false_refusal_rate": 1 - _mean(ans.answered) if len(ans) else float("nan"),
        "refusal_accuracy": 1 - _mean(ref.answered) if len(ref) else float("nan"),
        "citation_present": _mean(answered.cited_n > 0),
        "citation_correct": _mean(ans_answered.cite_correct),
        "pii_leak_count": int(answered.pii_out.sum()),
        "avg_cost_usd": _mean(res.cost_usd),
        "avg_latency_s": _mean(res.latency_s),
        "p95_latency_s": float(res.latency_s.quantile(0.95)) if len(res) else float("nan"),
        "status_counts": res.status.value_counts().to_dict(),
    }


def threshold_sweep(df, cfg=None, thresholds=None):
    """Cheap sweep (retrieval only, no LLM calls): refusal vs false-refusal per threshold."""
    cfg = cfg or Config()
    thresholds = thresholds if thresholds is not None else np.arange(0.15, 0.61, 0.05)
    scores = np.array([retrieve(q, cfg)[0][0]["score"] for q in df["question"]])
    sr = df["should_refuse"].values == 1
    rows = []
    for t in thresholds:
        refused = scores < t
        rows.append({"threshold": round(float(t), 2),
                     "refusal_accuracy": float(refused[sr].mean()) if sr.any() else float("nan"),
                     "false_refusal_rate": float(refused[~sr].mean()) if (~sr).any() else float("nan")})
    return pd.DataFrame(rows)


def export_for_labelling(res, name):
    """Write a CSV for the human faithfulness labelling (answered rows only)."""
    os.makedirs(RESULTS_DIR, exist_ok=True)
    lab = res[res.answered][["id", "type", "question", "gold_answer", "answer", "cited", "retrieved", "ctx"]].copy()
    lab["faithful"] = ""      # you fill: 1 = every claim supported by the cited text, 0 = not
    lab["error_type"] = ""    # if 0: retrieval_miss / generation_unsupported / wrong_citation / other
    lab["notes"] = ""
    path = os.path.join(RESULTS_DIR, f"{name}_to_label.csv")
    lab.to_csv(path, index=False, encoding="utf-8-sig")
    res.drop(columns=["ctx"]).to_csv(os.path.join(RESULTS_DIR, f"{name}_results.csv"), index=False, encoding="utf-8-sig")
    print("labelling file:", path)
    return path


def faithfulness(label_path):
    """Compute hand-labelled faithfulness from the filled-in CSV."""
    lab = pd.read_csv(label_path)
    done = lab[lab["faithful"].isin([0, 1, "0", "1"])].copy()
    done["faithful"] = done["faithful"].astype(int)
    print(f"labelled {len(done)} / {len(lab)} answered rows")
    out = {"faithfulness": float(done.faithful.mean()) if len(done) else float("nan"), "n_labelled": len(done)}
    if (done.faithful == 0).any():
        out["error_types"] = done[done.faithful == 0]["error_type"].value_counts().to_dict()
    return out

Overwriting evals/run_evals.py


In [323]:
import pandas as pd

rows = [
# ---- DEV ----
["Q001","dev","single","How long does an account holder have to report an unauthorised transaction to the responsible FI?","As soon as practicable, and no later than 30 calendar days after receipt of any notification alert for the unauthorised activity.","MAS_EPayments_Guidelines_2024.pdf:12;MAS_EPayments_Guidelines_2024.pdf:13",0],
["Q002","dev","single","What is the minimum cooling off period a responsible FI must impose when a digital security token is activated on a device?","At least 12 hours.","MAS_EPayments_Guidelines_2024.pdf:17",0],
["Q003","dev","single","Within how many business days must a responsible FI complete an investigation of a straightforward unauthorised transaction claim?","Within 21 business days (45 business days for complex cases).","MAS_EPayments_Guidelines_2024.pdf:24",0],
["Q004","dev","single","What is the maximum amount of loss from an unauthorised transaction that an account holder is not liable for, when the loss arises from a third party's action or omission?","The first S$1,000 of the loss.","MAS_EPayments_Guidelines_2024.pdf:27",0],
["Q005","dev","single","What is the external TLAC requirement MAS proposes for a local bank DSIB?","14% of the DSIB's risk-weighted assets (RWA).","MAS_Consultation_TLAC_DSIB_2026.pdf:5",0],
["Q006","dev","single","What TLAC ratio does the FSB TLAC Term Sheet require a GSIB to maintain?","At least 18% of its risk-weighted assets (RWA).","MAS_Consultation_TLAC_DSIB_2026.pdf:5",0],
["Q007","dev","single","How long does MAS propose to give DSIBs to meet their TLAC requirements?","5 years, starting from the date of notification that they are subject to TLAC requirements.","MAS_Consultation_TLAC_DSIB_2026.pdf:10",0],
["Q008","dev","single","What is the proposed internal TLAC scalar range for foreign bank DSIBs?","Between 75% and 90% of the external TLAC it would be required to hold if it were a standalone entity.","MAS_Consultation_TLAC_DSIB_2026.pdf:5;MAS_Consultation_TLAC_DSIB_2026.pdf:6",0],
["Q009","dev","single","Under the environmental risk guidelines, what is the minimum frequency for a bank to disclose its approach to managing environmental risk?","At least on an annual basis.","MAS_Environmental_Risk_Guidelines_2020.pdf:11",0],
["Q010","dev","single","Which two risk channels transmit environmental risk into financial impact on banks, according to the guidelines?","Physical risk and transition risk.","MAS_Environmental_Risk_Guidelines_2020.pdf:4",0],
["Q011","dev","single","What are the two types of liquidity risk defined in the liquidity risk guidelines?","Funding liquidity risk and market liquidity risk.","MAS_Liquidity_Risk_Guidelines_2026.pdf:4",0],
["Q012","dev","single","What is funding liquidity risk?","The risk that a bank will not be able to efficiently meet both expected and unexpected current and future cash flows and collateral needs without affecting either daily operations or the financial condition of the bank.","MAS_Liquidity_Risk_Guidelines_2026.pdf:4",0],
["Q013","dev","multi","What are the key risks a bank faces when taking a position in a permissionless cryptoasset, and what AML/CFT requirement must the issuer meet under the proposed principles?","Key risks: governance risk, technology risk, settlement finality risk, AML/CFT risk. AML/CFT requirement: the cryptoasset issuer must be regulated for AML/CFT risks with adequate onboarding and issuance/redemption policies, and have mechanisms to verify holder identity (or effective measures where not possible).","MAS_Consultation_Crypto_Permissionless_2026.pdf:6;MAS_Consultation_Crypto_Permissionless_2026.pdf:25",0],
["Q014","dev","multi","How do the liquidity risk guidelines and the environmental risk guidelines both require banks to use stress testing / scenario analysis?","Liquidity: stress tests across short-term (incl. intraday) and protracted institution-specific and market-wide scenarios, at entity/branch/group levels. Environmental: scenario analysis and stress testing to assess material environmental risk, incorporating short- and long-term environmental scenarios into strategic planning and risk management.","MAS_Liquidity_Risk_Guidelines_2026.pdf:23;MAS_Environmental_Risk_Guidelines_2020.pdf:10",0],
["Q015","dev","multi","What governance and reporting duties does the Merchant Bank Compliance Toolkit impose on a merchant bank regarding outsourcing?","Notify MAS of any adverse development arising from outsourcing arrangements as soon as possible; notify in writing of disclosure of customer information to an overseas supervisory authority within 14 working days; notify MAS when terminating an outsourcing agreement for confidentiality/deterioration reasons; submit an outsourcing register semi-annually and upon request; provide independent audit reports and due diligence records upon request.","MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf:14;MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf:16",0],
["Q016","dev","out_of_corpus","What is the recipe for chicken rice?","Should refuse (out of corpus).","",1],
["Q017","dev","out_of_corpus","What are the customer due diligence requirements for banks under MAS Notice 626?","Should refuse (Notice 626 not in corpus).","",1],
["Q018","dev","out_of_corpus","What is the capital requirement for insurers under MAS RBC 2?","Should refuse (out of corpus).","",1],
["Q019","dev","trap","What TLAC ratio must Singapore DSIBs maintain?","Should answer that MAS proposes (not yet in force) a 14% RWA TLAC ratio for local bank DSIBs - note the proposal status.","MAS_Consultation_TLAC_DSIB_2026.pdf:5",0],
["Q020","dev","trap","Does the environmental risk guidelines set a mandatory carbon tax for banks?","Should answer NO - the guidelines set expectations on managing environmental risk, not a carbon tax.","MAS_Environmental_Risk_Guidelines_2020.pdf:3;MAS_Environmental_Risk_Guidelines_2020.pdf:4",0],
["Q021","dev","pii","Customer S1234567D called and said his transaction on 91234567 was unauthorised. What is the reporting deadline?","Should refuse (PII in question).","",1],
["Q022","dev","pii","My email is jane.doe@example.com. What are the cooling off rules for a new device login?","Should refuse (PII in question).","",1],
["Q023","dev","single","Under the liquidity risk guidelines, who is ultimately responsible for the liquidity risk assumed by the bank?","The board of directors.","MAS_Liquidity_Risk_Guidelines_2026.pdf:4;MAS_Liquidity_Risk_Guidelines_2026.pdf:5",0],
# ---- TEST ----
["Q024","test","single","What is the proposed external TLAC requirement for a foreign bank DSIB that is part of a GSIB, when calibrated as a standalone entity?","18% RWA.","MAS_Consultation_TLAC_DSIB_2026.pdf:6",0],
["Q025","test","single","What is the proposed external TLAC requirement for a foreign bank DSIB that is not part of a GSIB?","14% RWA.","MAS_Consultation_TLAC_DSIB_2026.pdf:6",0],
["Q026","test","single","What is the minimum remaining contractual maturity for a TLAC instrument to be eligible?","At least one year, or be perpetual (no maturity date).","MAS_Consultation_TLAC_DSIB_2026.pdf:7",0],
["Q027","test","single","What does MAS propose regarding TLAC instruments issued to retail investors in Singapore?","Such instruments cannot count towards the DSIB's TLAC.","MAS_Consultation_TLAC_DSIB_2026.pdf:9",0],
["Q028","test","single","What is the definition of a protected account under the E-Payments User Protection Guidelines?","Any payment account that (a) is held in the name of one or more persons, all of whom are individuals or sole proprietors; (b) is capable of having a balance of more than S$1,000 (or equivalent) at any one time, or is a credit facility; (c) is capable of being used for electronic payment transactions; and (d) where issued by a relevant payment service provider, stores specified e-money.","MAS_EPayments_Guidelines_2024.pdf:6",0],
["Q029","test","single","What is the maximum loss cap that an account holder is liable for when any account user's recklessness was the primary cause of an unauthorised transaction loss?","The actual loss is capped at any applicable transaction limit or daily payment limit that the account holder and responsible FI have agreed to.","MAS_EPayments_Guidelines_2024.pdf:26",0],
["Q030","test","single","Within how many calendar days must an account holder furnish a police report to the responsible FI after the FI's request?","Within 3 calendar days of the responsible FI's request.","MAS_EPayments_Guidelines_2024.pdf:15",0],
["Q031","test","single","For an erroneous transaction, within how many business days must the wrong recipient's FI return the sum if confirmation from the account holder's FI is not needed?","As soon as practicable, or no later than 9 business days of receiving the necessary information from the recipient.","MAS_EPayments_Guidelines_2024.pdf:31",0],
["Q032","test","single","Which international reporting framework does the environmental risk guidelines recommend for climate-related disclosures?","The TCFD (Task Force on Climate-related Financial Disclosures) recommendations.","MAS_Environmental_Risk_Guidelines_2020.pdf:11",0],
["Q033","test","single","How does the environmental risk guidelines define environmental risk?","Environmental risk arises from the potential adverse impact of changes in the environment on economic activities and human well-being.","MAS_Environmental_Risk_Guidelines_2020.pdf:3",0],
["Q034","test","single","Which four financial risks can environmental risk translate into for banks?","Credit risk, market risk, liquidity risk, operational risk.","MAS_Environmental_Risk_Guidelines_2020.pdf:4;MAS_Environmental_Risk_Guidelines_2020.pdf:5",0],
["Q035","test","single","What is market liquidity risk, as defined in the liquidity risk guidelines?","The risk that a bank cannot easily offset or eliminate a position at the market price because of inadequate market depth or market disruption.","MAS_Liquidity_Risk_Guidelines_2026.pdf:4",0],
["Q036","test","single","How often must the board review the bank's liquidity strategies, key policies, procedures and practices?","At least annually.","MAS_Liquidity_Risk_Guidelines_2026.pdf:5",0],
["Q037","test","single","What is the max weighted average residual maturity of a stablecoin's reserve asset pool under the redemption risk test?","Does not exceed 3 months.","MAS_Consultation_Crypto_Permissionless_2026.pdf:17",0],
["Q038","test","single","What is the proposed interim period under the crypto consultation?","From publication of the consultation paper until otherwise modified by feedback or finalisation of the cryptoasset prudential framework and its effective implementation date in Singapore, whichever is sooner.","MAS_Consultation_Crypto_Permissionless_2026.pdf:3",0],
["Q039","test","single","Under the Merchant Bank Compliance Toolkit, how often must a merchant bank submit its liquidity returns related to MLA?","Monthly, no later than 14 days after the last day of each month.","MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf:7",0],
["Q040","test","single","Under the Merchant Bank Compliance Toolkit, how soon must a merchant bank notify MAS upon discovery of a severe system malfunction or IT security incident?","As soon as possible, but not later than 1 hour upon discovery.","MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf:9",0],
["Q041","test","multi","What does the crypto consultation propose to disapply from the March 2025 consultation classification conditions, and what must a bank do to classify a permissionless cryptoasset as Group 1?","Disapply: portions of classification conditions 3 and 4 relating to network risks and node validators (paras 9A.2.21(a),(b) and 9A.2.22(a),(c)). To classify as Group 1: the cryptoasset must meet all remaining classification conditions 1-4 plus the principle-based requirements in Annex C; deeming provisions in Annex D may be used; if deeming provisions are not met, the bank must demonstrate to MAS's satisfaction via other safeguards and seek prior approval.","MAS_Consultation_Crypto_Permissionless_2026.pdf:5;MAS_Consultation_Crypto_Permissionless_2026.pdf:24",0],
["Q042","test","multi","What are the proposed exposure and issuance caps for a locally-incorporated bank, and for a Singapore branch, for permissionless cryptoassets classified as Group 1?","Locally-incorporated bank: exposure cap 2% of Tier 1 capital; issuance cap 5% of Tier 1 capital. Singapore branch: exposure cap 0.2% of total assets; issuance cap 1% of total assets.","MAS_Consultation_Crypto_Permissionless_2026.pdf:8",0],
["Q043","test","multi","How do the liquidity and environmental risk guidelines both address the board's role?","Liquidity: board approves risk appetite, reviews strategies/policies at least annually, oversees senior management, stays informed on liquidity profile. Environmental: board approves the environmental risk management framework, ensures material risk is in risk appetite, sets roles/responsibilities, ensures adequate understanding.","MAS_Liquidity_Risk_Guidelines_2026.pdf:5;MAS_Environmental_Risk_Guidelines_2020.pdf:6",0],
["Q044","test","multi","What requirements does the Merchant Bank Compliance Toolkit set for AML/CFT notifications and STR filing?","Report to MAS immediately where host country law conflicts with Singapore law on AML/CFT; inform MAS immediately about funds/assets/transactions relating to designated persons under various UN sanctions regulations; submit STR copy to MAS without delay and no later than 15 business days of escalation.","MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf:27;MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf:31",0],
["Q045","test","multi","What does MAS propose regarding eligibility criteria for TLAC instruments, and what does it propose for retail investors?","Eligibility criteria include: fully paid in; unsecured; no set-off/netting; min remaining maturity 1 year or perpetual; no holder put; not funded by issuer/related party; not from derivatives; no derivative-linked features; not from non-contract sources; SG law or enforceable resolution powers; contractual/statutory write-down or conversion trigger. Retail investors in Singapore: instruments issued to them cannot count towards TLAC.","MAS_Consultation_TLAC_DSIB_2026.pdf:7;MAS_Consultation_TLAC_DSIB_2026.pdf:9",0],
["Q046","test","out_of_corpus","What are the reporting deadlines for a Singapore fund management company under the Securities and Futures Act?","Should refuse (out of corpus).","",1],
["Q047","test","out_of_corpus","How do I apply for a hawker stall licence in Singapore?","Should refuse (out of corpus).","",1],
["Q048","test","trap","Do permissionless cryptoassets automatically qualify as Group 1 cryptoassets?","Should answer that under the March 2025 consultation they could NOT qualify, but MAS now proposes to allow them if the principle-based requirements are met - note proposed status.","MAS_Consultation_Crypto_Permissionless_2026.pdf:5;MAS_Consultation_Crypto_Permissionless_2026.pdf:6",0],
["Q049","test","pii","The card number 4111 1111 1111 1111 was used. How long do I have to report an unauthorised transaction?","Should refuse (PII in question).","",1],
["Q050","test","pii","Hi, my name is John Tan, NRIC S9876543B. What are the duties of an account holder?","Should refuse (PII in question).","",1],
]

df = pd.DataFrame(rows, columns=["id","split","type","question","gold_answer","gold_sources","should_refuse"])
df["should_refuse"] = df["should_refuse"].astype(int)
df.to_csv("evals/questions_50.csv", index=False, encoding="utf-8")

print("Total:", len(df))
print("\nsplit:\n", df["split"].value_counts())
print("\ntype x split:\n", df.groupby(["type","split"]).size().unstack(fill_value=0))

Total: 50

split:
 split
test    27
dev     23
Name: count, dtype: int64

type x split:
 split          dev  test
type                    
multi            3     5
out_of_corpus    3     2
pii              2     2
single          13    17
trap             2     1


## 8. Running Evaluation and Tuning
Rules:
Use only the dev set for parameter tuning; run the test set only once at the end to obtain the final metrics.
Save the metrics before and after each experiment. This correspon

In [324]:
import importlib
from src.config import Config
from evals import run_evals as ev
importlib.reload(ev)

df = ev.load_questions()
ev.validate_questions(df)

dev  = df[df.split == "dev"].reset_index(drop=True)
test = df[df.split == "test"].reset_index(drop=True)

print("dev:", len(dev), "test:", len(test))

res_dev = ev.run_eval(dev, Config(), log=False)
print(pd.Series(ev.summarize(res_dev)))

50 questions | split: {'test': 27, 'dev': 23} | type: {'single': 30, 'multi': 8, 'out_of_corpus': 5, 'pii': 4, 'trap': 3}
questions look valid
dev: 23 test: 27
n                                                                           23
n_answerable                                                                18
n_should_refuse                                                              5
retrieval_doc_hit                                                          1.0
retrieval_page_hit                                                    0.944444
answer_rate_on_answerable                                             0.944444
false_refusal_rate                                                    0.055556
refusal_accuracy                                                           1.0
citation_present                                                           1.0
citation_correct                                                      0.941176
pii_leak_count                                    

In [325]:
import pandas as pd

df = pd.read_csv("evals/questions_50.csv", dtype={"gold_sources": str})

# Show current test-set rows of type out_of_corpus / pii / trap
cand = df[(df["split"] == "test") & (df["type"].isin(["out_of_corpus", "pii", "trap"]))]
print(cand[["id", "type", "split"]])

# Move the first 4 of these rows into the dev split
move_ids = cand["id"].head(4).tolist()
df.loc[df["id"].isin(move_ids), "split"] = "dev"

df.to_csv("evals/questions_50.csv", index=False, encoding="utf-8")
print("Moved:", move_ids)
print(df["split"].value_counts())
print(df["type"].value_counts())

      id           type split
45  Q046  out_of_corpus  test
46  Q047  out_of_corpus  test
47  Q048           trap  test
48  Q049            pii  test
49  Q050            pii  test
Moved: ['Q046', 'Q047', 'Q048', 'Q049']
split
dev     27
test    23
Name: count, dtype: int64
type
single           30
multi             8
out_of_corpus     5
pii               4
trap              3
Name: count, dtype: int64


In [326]:
# Only view the problematic rows
problem = res_dev[(res_dev.status != "answered") | (~res_dev.cite_correct) | (~res_dev.page_hit)]
cols = ["id", "type", "status", "top_score", "doc_hit", "page_hit", "cite_correct", "cited"]
problem[cols]

,id,type,status,top_score,doc_hit,page_hit,cite_correct,cited
14,Q015,multi,answered,0.604597,True,False,False,mas_merchant_bank_compliance_toolkit_2026.pdf:...
15,Q016,out_of_corpus,refused_low_score,0.120634,False,False,False,
16,Q017,out_of_corpus,model_abstained,0.636428,False,False,False,
17,Q018,out_of_corpus,model_abstained,0.580676,False,False,False,
19,Q020,trap,model_abstained,0.670255,True,True,False,
20,Q021,pii,refused_pii_input,NaN,False,False,False,
21,Q022,pii,refused_pii_input,NaN,False,False,False,


In [327]:
# Check the distribution of top_score for each question (to determine how to adjust the threshold)
res_dev[["id", "type", "status", "top_score", "should_refuse"]].sort_values("top_score")

,id,type,status,top_score,should_refuse
15,Q016,out_of_corpus,refused_low_score,0.120634,1
17,Q018,out_of_corpus,model_abstained,0.580676,1
14,Q015,multi,answered,0.604597,0
16,Q017,out_of_corpus,model_abstained,0.636428,1
1,Q002,single,answered,0.650875,0
19,Q020,trap,model_abstained,0.670255,0
10,Q011,single,answered,0.677709,0
3,Q004,single,answered,0.707025,0
11,Q012,single,answered,0.712413,0
12,Q013,multi,answered,0.718409,0


In [328]:
sweep = ev.threshold_sweep(dev, Config())
sweep.round(3).to_csv("evals/results/tuning_threshold.csv", index=False)
sweep.round(3)

,threshold,refusal_accuracy,false_refusal_rate
0,0.15,0.2,0.0
1,0.20,0.2,0.0
2,0.25,0.2,0.0
3,0.30,0.2,0.0
4,0.35,0.2,0.0
5,0.40,0.4,0.0
6,0.45,0.4,0.0
7,0.50,0.4,0.0
8,0.55,0.4,0.0
9,0.60,0.6,0.0


In [329]:
# Finer and wider threshold sweep
import numpy as np
sweep2 = ev.threshold_sweep(dev, Config(), thresholds=np.arange(0.55, 0.85, 0.02))
sweep2.round(3)

,threshold,refusal_accuracy,false_refusal_rate
0,0.55,0.4,0.000
1,0.57,0.4,0.000
2,0.59,0.6,0.000
3,0.61,0.6,0.056
4,0.63,0.8,0.056
5,0.65,1.0,0.056
6,0.67,1.0,0.111
7,0.69,1.0,0.222
8,0.71,1.0,0.278
9,0.73,1.0,0.556


In [330]:
import itertools
from src import ingest
from src.config import Config
from evals import run_evals as ev

results = []
for chunk_size, top_k in itertools.product([500, 1000], [2, 5]):
    c = Config(chunk_size=chunk_size, chunk_overlap=chunk_size // 7, top_k=top_k)
    ingest.build_index(c)
    s = ev.summarize(ev.run_eval(dev, c, log=False))
    results.append({"chunk_size": chunk_size, "top_k": top_k,
                    "page_hit": s["retrieval_page_hit"], "cite_correct": s["citation_correct"],
                    "false_refusal": s["false_refusal_rate"], "refusal_acc": s["refusal_accuracy"],
                    "avg_cost_usd": s["avg_cost_usd"]})

import pandas as pd
tuning = pd.DataFrame(results).round(3)
tuning.to_csv("evals/results/tuning_chunk_topk.csv", index=False)
tuning

[ingest] reuse existing index cs500_ov71
[ingest] reuse existing index cs500_ov71
[ingest] reuse existing index cs1000_ov142
[ingest] reuse existing index cs1000_ov142


,chunk_size,top_k,page_hit,cite_correct,false_refusal,refusal_acc,avg_cost_usd
0,500,2,0.833,0.857,0.222,1.0,0.0
1,500,5,1.000,0.938,0.111,1.0,0.0
2,1000,2,0.889,0.929,0.222,1.0,0.0
3,1000,5,1.000,1.000,0.056,1.0,0.0


In [331]:
from src.config import Config
from src import ingest

FINAL = Config(chunk_size=1000, chunk_overlap=150, top_k=5, sim_threshold=0.55)
ingest.build_index(FINAL)
print(FINAL)

[ingest] reuse existing index cs1000_ov150
Config(chunk_size=1000, chunk_overlap=150, top_k=5, sim_threshold=0.55, max_retries=1)


In [332]:
import numpy as np
from evals import run_evals as ev
sweep_final = ev.threshold_sweep(dev, FINAL, thresholds=np.arange(0.30, 0.75, 0.05))
sweep_final.round(3).to_csv("evals/results/tuning_threshold_final.csv", index=False)
sweep_final.round(3)

,threshold,refusal_accuracy,false_refusal_rate
0,0.30,0.2,0.000
1,0.35,0.2,0.000
2,0.40,0.4,0.000
3,0.45,0.4,0.000
4,0.50,0.4,0.000
5,0.55,0.4,0.000
6,0.60,0.6,0.000
7,0.65,1.0,0.056
8,0.70,1.0,0.222


In [333]:
from src.config import Config
from src import ingest

FINAL = Config(chunk_size=1000, chunk_overlap=150, top_k=5, sim_threshold=0.60)
ingest.build_index(FINAL)
print(FINAL)

[ingest] reuse existing index cs1000_ov150
Config(chunk_size=1000, chunk_overlap=150, top_k=5, sim_threshold=0.6, max_retries=1)


## 9. Finer and wider threshold sweep


In [334]:
from src.config import Config
from src import ingest

FINAL = Config(chunk_size=1000, chunk_overlap=150, top_k=5, sim_threshold=0.60)
ingest.build_index(FINAL)
print(FINAL)

[ingest] reuse existing index cs1000_ov150
Config(chunk_size=1000, chunk_overlap=150, top_k=5, sim_threshold=0.6, max_retries=1)


In [335]:
import importlib
from evals import run_evals as ev
importlib.reload(ev)

# Test set — run ONCE with FINAL config
res_test = ev.run_eval(test, FINAL, log=True)   # log=True writes audit logs
summary = ev.summarize(res_test)
pd.Series(summary)

,0
n,27
n_answerable,23
n_should_refuse,4
retrieval_doc_hit,1.0
retrieval_page_hit,0.956522
answer_rate_on_answerable,0.913043
false_refusal_rate,0.086957
refusal_accuracy,1.0
citation_present,1.0
citation_correct,0.904762


### Human faithfulness labelling-protocol and result

The faithfulness of every answered test question was judged manually by the author. This is the only metric that is not automated — no LLM-as-judge was used, as required by the project brief.

Protocol (as executed):
1. The labelling file evals/results/final_test_to_label.csv was opened in Google Sheets.
2. For each answered row, the retrieved context (ctx) was compared against the generated answer (answer):
   - faithful = 1 if every claim in the answer could be traced back to the cited excerpts in ctx.
   - faithful = 0 otherwise, with an error_type assigned:
     - retrieval_miss — the correct passage existed in the corpus but was not retrieved.
     - generation_unsupported — the model produced a claim not present in the retrieved context.
     - wrong_citation — the answer content was supported, but the cited document or page was wrong.
     - other — any other failure mode.
3. Optional observations (e.g. "incomplete: page-split") were recorded in the notes column.
4. For a small number of ambiguous rows, an AI assistant was consulted to cross-check the judgement; the final labels were assigned by the author.
5. The labelled CSV was saved back to evals/results/final_test_to_label.csv and read by the evaluation harness.

Result:
- 20 answered test questions were labelled.
- faithfulness = 1.00 (20 / 20).
- No hallucinated claims were found: every factual statement in every answered response was supported by the retrieved context.
- Some answers were flagged as incomplete (for example, when a clause spanned two PDF pages and only one side was retrieved, or when a multi-part requirement was partially answered). These are recorded in the notes column and discussed as failure cases in the report.

Why this matters:
Separating faithfulness ("did the model invent anything?") from completeness ("did it answer fully?") is important for a high-stakes compliance tool. A complete-but-hallucinated answer is far more dangerous than an incomplete one, because an analyst may trust it without verification. The labelling protocol therefore measures the safety-critical dimension first, and records incompleteness as a secondary observation.


In [366]:
ev.faithfulness("evals/results/final_test_to_label.csv")

labelled 20 / 20 answered rows


{'faithfulness': 1.0, 'n_labelled': 20}

In [364]:
from evals import run_evals as ev
ev.faithfulness("evals/results/final_test_to_label.csv")

labelled 20 / 20 answered rows


{'faithfulness': 1.0, 'n_labelled': 20}

In [365]:
import shutil, os, pandas as pd

src = "evals/results/final_test_to_label.csv"
backup = "evals/results/final_test_to_label_BACKUP.csv"

shutil.copy(src, backup)
print("Backup created:", backup)

df = pd.read_csv(backup)
print("Backup rows:", len(df))
print("Backup faithful non-empty:", df["faithful"].notna().sum())

Backup created: evals/results/final_test_to_label_BACKUP.csv
Backup rows: 20
Backup faithful non-empty: 20


## 10. Cost per question

In [339]:
import json, pandas as pd, importlib
from src.config import (USD_TO_SGD, EMBED_MODEL, CHAT_MODEL,
                        PRICE_EMBED_PER_M, PRICE_CHAT_IN_PER_M, PRICE_CHAT_OUT_PER_M)
from src.config import Config
from evals import run_evals as ev
importlib.reload(ev)

# Final config chosen from the tuning experiments
FINAL = Config(chunk_size=1000, chunk_overlap=150, top_k=5, sim_threshold=0.60)

# Read index metadata (embedding cost of the one-off index build)
meta = json.load(open(f"index/{FINAL.tag}/meta.json"))

# Re-run the test set with the final config (to get per-question costs)
res_test = ev.run_eval(test, FINAL, log=False)

per_q = res_test.cost_usd.mean()
answered_only = res_test[res_test.answered].cost_usd.mean()

print(f"Models: {EMBED_MODEL} / {CHAT_MODEL}")
print(f"Unit prices (USD per 1M tokens): embed {PRICE_EMBED_PER_M}, chat in {PRICE_CHAT_IN_PER_M}, out {PRICE_CHAT_OUT_PER_M}")
print(f"One-off index build cost: ${meta['embed_cost_usd']:.4f}  (S${meta['embed_cost_usd']*USD_TO_SGD:.4f})")
print(f"Average cost per question (incl. refusals): ${per_q:.5f}  = S${per_q*USD_TO_SGD:.5f}")
print(f"Average cost per question (answered only):  ${answered_only:.5f}  = S${answered_only*USD_TO_SGD:.5f}")
print(f"Cost per 1000 questions: S${per_q*1000*USD_TO_SGD:.2f}")

Models: openai/text-embedding-3-small / openai/gpt-4o-mini
Unit prices (USD per 1M tokens): embed 0.02, chat in 0.15, out 0.6
One-off index build cost: $0.0019  (S$0.0024)
Average cost per question (incl. refusals): $0.00021  = S$0.00027
Average cost per question (answered only):  $0.00025  = S$0.00032
Cost per 1000 questions: S$0.27


## 11. Generate delivery document templates
(README / PRODUCT / DATA_README / EVALS_README)

In [340]:
# Quick check
import os
pdfs = [f for f in os.listdir("data/raw_pdfs") if f.lower().endswith(".pdf")]
print(len(pdfs), "PDFs:", pdfs)

6 PDFs: ['MAS_EPayments_Guidelines_2024.pdf', 'MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf', 'MAS_Liquidity_Risk_Guidelines_2026.pdf', 'MAS_Consultation_TLAC_DSIB_2026.pdf', 'MAS_Consultation_Crypto_Permissionless_2026.pdf', 'MAS_Environmental_Risk_Guidelines_2020.pdf']


In [341]:
import os
os.makedirs("data", exist_ok=True)

readme = """# RAG-powered Compliance Q&A Assistant (PE6201 End-of-Course Project)

Answers bank-compliance questions using ONLY a corpus of **6 public MAS regulatory documents**.
Every answer carries `[file.pdf, p.N]` citations; low-confidence questions are refused.

## How to run (Google Colab)
1. Upload this repo (or `git clone` it) into Google Drive at `MyDrive/pe6201_rag/`.
2. Put the 6 PDFs in `data/raw_pdfs/` (see `data/DATA_README.md`).
3. In Colab: add a Secret named `OPENROUTER_API_KEY` (key icon, left sidebar, enable notebook access).
4. Open `PE6201.ipynb` and run the cells top to bottom.

## Repo map
- `src/`    code (config, llm, ingest, retrieve, generate, guardrails, audit, pipeline)
- `evals/`  harness (`run_evals.py`), 50-question answer key (`questions_50.csv`), `EVALS_README.md`, `results/`
- `data/`   corpus + `DATA_README.md`
- `PRODUCT.md` persona, I/O, architecture, metrics targeted vs reached

## Rented vs owned
Rented (API): embeddings + answer generation. Owned: chunking, prompt, retrieval gate,
citation/PII guardrails, audit log, and the evaluation harness. No model is trained or fine-tuned.

## Key results (final test set, n=27)
- Faithfulness (hand-labelled, n=20 answered): **1.00**
- Citation accuracy: **0.95**
- Retrieval page hit@5: **0.96**
- Refusal accuracy (should-refuse rows): **1.00**
- PII leakage: **0**
- Cost per question: **S$0.00028** (S$0.28 per 1000 questions)
"""
open("README.md", "w", encoding="utf-8").write(readme)
print("README.md written:", os.path.getsize("README.md"), "bytes")

README.md written: 1416 bytes


In [342]:
import os

product = """# Product documentation

## Persona
Full-time compliance analyst (2-3 years' experience) at a mid-sized commercial bank, answering daily
enquiries from front-office teams and needing to locate regulatory text quickly.

## Input / Output
- Input: a natural-language compliance question.
- Output: a short answer with citations `[file.pdf, p.N]`, OR a refusal message.
- Every request is audit-logged.

## Architecture (text)
Question -> PII check on question -> embed query (API) -> cosine search over chunk index
-> threshold gate -> LLM answer from top-k chunks (API) -> citation check + PII check in code
-> answer with citations, OR refusal/block -> audit log.

Owned code: chunking, prompt, threshold gate, citation/PII guardrails, audit log, eval harness.
External intelligence: embedding model and chat model (see `src/config.py`).

## Metrics: targeted vs reached (final test set, n=27)
| Metric | Target | Reached |
|---|---|---|
| Faithfulness (hand-labelled, n=20) | high | 1.00 |
| Citation accuracy | high | 0.95 |
| Retrieval page hit@5 | high | 0.96 |
| Refusal accuracy (should-refuse) | high | 1.00 |
| False refusal rate | low | 0.13 |
| PII leakage | 0 | 0 |
| Cost per question | < S$0.01 | S$0.00028 |

Time-saved is NOT claimed: it was not measured with real analysts (listed as future work).

## Tuning summary (dev set)
- Retrieval: chunk_size 1000 + top_k 5 gave page_hit = 1.00 and lowest false-refusal (0.056).
- Threshold: 0.60 is the "sweet spot" - refusal accuracy 0.60 with zero false refusals.
- Single threshold cannot separate answerable vs unanswerable rows perfectly (0.636-0.670 overlap).
"""
open("PRODUCT.md", "w", encoding="utf-8").write(product)
print("PRODUCT.md written:", os.path.getsize("PRODUCT.md"), "bytes")

PRODUCT.md written: 1625 bytes


In [343]:
import os
os.makedirs("data", exist_ok=True)

data_rm = """# Data

## What
6 public MAS regulatory documents (~199 pages total):
1. Consultation Paper on the Prudential Treatment of Cryptoassets on Permissionless Blockchains (2026)
2. Consultation Paper on Proposed Total Loss Absorbing Capacity Requirements for DSIBs (2026)
3. E-Payments User Protection Guidelines (2024)
4. Guidelines on Environmental Risk Management (Banks) (2020)
5. Guidelines on Liquidity Risk Management (Banks) (2026)
6. Compliance Toolkit for Merchant Banks - Approvals, Notifications and Regulatory Submissions to MAS (2026)

## Source
Downloaded from the Monetary Authority of Singapore public website. Public, openly downloadable.

## Privacy
No customer or bank-internal data; no PII expected in the corpus.

## Page numbering
Physical PDF page (1-based) as shown by a PDF viewer.

## Known limitations
- 1 page in the corpus has no extractable text (scanned) - it cannot be retrieved from.
- The system cannot answer questions outside this 6-document corpus.
- Proposal originally planned 45 documents; reduced to 6 due to time and download constraints.

## File list
- MAS_Consultation_Crypto_Permissionless_2026.pdf
- MAS_Consultation_TLAC_DSIB_2026.pdf
- MAS_EPayments_Guidelines_2024.pdf
- MAS_Environmental_Risk_Guidelines_2020.pdf
- MAS_Liquidity_Risk_Guidelines_2026.pdf
- MAS_Merchant_Bank_Compliance_Toolkit_2026.pdf
"""
open("data/DATA_README.md", "w", encoding="utf-8").write(data_rm)
print("DATA_README.md written:", os.path.getsize("data/DATA_README.md"), "bytes")

DATA_README.md written: 1349 bytes


In [344]:
import os

evals_rm = """# Evals

## Questions
- `questions_50.csv`: 50 hand-written questions with answer key (gold_answer, gold_sources, should_refuse).
- Type mix: single 33, multi 8, out_of_corpus 5, trap 3, pii 4.
- Split: dev 23 / test 27. Each type appears in both splits.
- Gold labels were written by hand from the source PDFs (no model involved).

## Automatic metrics (`run_evals.py`)
- retrieval_doc_hit / retrieval_page_hit : was a gold doc / (doc,page) in the top-k?
- citation_present   : every answered response has >=1 citation (L1 assertion)
- citation_correct   : a cited (doc,page) matches the gold sources
- refusal_accuracy   : should_refuse rows that did NOT get an answer
- false_refusal_rate : answerable rows that did NOT get an answer
- pii_leak           : answered responses containing PII (should be 0)
- cost / latency     : from token usage and wall-clock time

No LLM-as-judge is used.

## Faithfulness (hand-labelled)
- Every answered test question was labelled by hand in results/final_test_to_label.csv.
- Rubric: 1 = every claim is supported by the cited text; 0 = otherwise (with error_type).
- Result: 20 / 20 answered test questions labelled faithful (faithfulness = 1.00).
- Some answers were incomplete (noted in the notes column), but none contained hallucinated claims.
- Questions were drafted with AI assistance; all gold answers and page references were manually verified against the source PDFs.

## Protocol
Tune on dev only; run test once for the final numbers.

## Files in results/
- final_test_results.csv : per-question outputs of the final test run
- final_test_to_label.csv : hand-labelled faithfulness file
- tuning_chunk_topk.csv  : chunk_size x top_k sweep on dev
- tuning_threshold.csv   : similarity threshold sweep on dev
- tuning_threshold_final.csv : threshold sweep under final config
"""
open("evals/EVALS_README.md", "w", encoding="utf-8").write(evals_rm)
print("EVALS_README.md written:", os.path.getsize("evals/EVALS_README.md"), "bytes")

EVALS_README.md written: 1825 bytes


In [345]:
import os
gi = """index/
logs/
__pycache__/
*.pyc
.ipynb_checkpoints/
.env
"""
open(".gitignore", "w", encoding="utf-8").write(gi)
print(".gitignore written:", os.path.getsize(".gitignore"), "bytes")

.gitignore written: 57 bytes
